# GLASS Interactive SI Shift

Display interactive latest-observed-count vs future SI-shift plots, with GLASS pathogen-antibiotic pairs highlighted.

In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipelines import (
    build_glass_match_table,
    build_novel_prediction_table,
    build_si_analysis_paths,
    load_or_build_interactive_rank,
    load_si_and_clean_glass_list,
    plot_future_si_vs_si_shift,
    plot_latest_si_vs_future_si,
    summarize_pathogen_coverage,
)
from amr.si import plot_interactive_latest_observed_count_vs_future_si_shift

## Parameters

In [21]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PARQUET_OUTPUT_DIR = OUTPUT_DIR / "parquets"
FIGURE_OUTPUT_DIR = OUTPUT_DIR / "figures"
PARQUET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PARQUET_OUTPUT_DIR
ASSETS_DIR = PROJECT_ROOT / "assets"

ANTIBIOTIC_ANALYSIS_LEVEL = "antibiotic"  # antibiotic, Class, Simplified categories, High level categories, Origin
SI_PATHS = build_si_analysis_paths(PROJECT_ROOT, ANTIBIOTIC_ANALYSIS_LEVEL)
YEARLY_SCREENED_PATH = PARQUET_OUTPUT_DIR / "yearly_amr_screened_pairs.parquet"
YEARLY_SI_PATH = SI_PATHS["yearly_si"]
RANK_OUTPUT_PATH = SI_PATHS["rank_output"]

GLASS_XLSX_PATH = ASSETS_DIR / "GLASS list.xlsx"
GLASS_CLEANED_PATH = ASSETS_DIR / "glass_amr_cleaned.csv"
BACTERIA_SYNONYM_PATH = ASSETS_DIR / "bacteria_synonym_lookup.csv"
ANTIBIOTIC_SYNONYM_PATH = ASSETS_DIR / "antibiotic_synonym_lookup.csv"
ANTIBIOTIC_COARSENING_PATH = ASSETS_DIR / "antibiotic_coarsening_lookup.csv"

FIT_START_YEAR = 2000
FIT_END_YEAR = 2023
CURVE_FIT_END_YEAR = 2025
NUM_FUTURE_YEARS = 3
LOESS_FRAC = 0.6
LOESS_DEGREE = 1
TAIL_DAMPING_YEARS = None

MIN_LAST_AVAILABLE_COUNT = 10
MIN_COUNT_ABS_PCT_ERROR_MEAN = 0
MAX_COUNT_ABS_PCT_ERROR_MEAN = 50
MIN_COUNTS_BACKGROUND = 20
MIN_ABS_DELTA_SI = 0.01
MIN_SI_DELTA = None

DISPLAY_NAME = True
MAX_TEXT_LABELS = 300
WIDTH = 1400
HEIGHT = 900

## Load SI And Clean GLASS List

In [22]:
si_df, glass_pairs_df = load_si_and_clean_glass_list(
    YEARLY_SI_PATH,
    GLASS_XLSX_PATH,
    bacteria_synonym_path=BACTERIA_SYNONYM_PATH,
    antibiotic_synonym_path=ANTIBIOTIC_SYNONYM_PATH,
    antibiotic_coarsening_path=ANTIBIOTIC_COARSENING_PATH,
    glass_cleaned_path=GLASS_CLEANED_PATH,
)

si_source_df = pd.read_parquet(YEARLY_SCREENED_PATH)
si_pathogen_summary_df = summarize_pathogen_coverage(
    source_df=si_source_df,
    modeled_df=si_df,
    source_label="screened AMR input from notebook 01",
    modeled_label="filtered SI dataset from notebook 02",
)
display(si_pathogen_summary_df.query("status != 'present_in_both'"))
display(glass_pairs_df.head(20))

Loaded SI rows: 6,209 from /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/outputs/parquets/yearly_si.parquet
Pathogens: 19; antibiotics/levels: 60
Saved cleaned GLASS rows: 89 to /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/glass_amr_cleaned.csv
screened AMR input from notebook 01 pathogens: 48
filtered SI dataset from notebook 02 pathogens: 19
Pathogens present in screened AMR input from notebook 01 but absent from filtered SI dataset from notebook 02: 29
Aggregatibacter actinomycetemcomitans, Bacillus anthracis, Bordetella pertussis, Borrelia burgdorferi, Brucella melitensis, Burkholderia cepacia, Burkholderia pseudomallei, Chlamydia pneumoniae, Chlamydia trachomatis, Clostridium perfringens, Coxiella burnetii, Cutibacterium acnes, Francisella tularensis, Legionella pneumophila, Moraxella catarrhalis, Mycobacterium abscessus, Mycobacterium avium, Mycobacterium leprae, 

,pathogen,source_rows,source_publications,source_pairs,modeled_rows,modeled_publications,modeled_pairs,in_source,in_modeled,status
0,Aggregatibacter actinomycetemcomitans,97,114,21,0.0,0.0,0.0,True,False,missing_from_modeled
1,Bacillus anthracis,78,62,47,0.0,0.0,0.0,True,False,missing_from_modeled
2,Bordetella pertussis,71,97,25,0.0,0.0,0.0,True,False,missing_from_modeled
3,Borrelia burgdorferi,41,43,23,0.0,0.0,0.0,True,False,missing_from_modeled
4,Brucella melitensis,124,170,31,0.0,0.0,0.0,True,False,missing_from_modeled
5,Burkholderia cepacia,262,403,48,0.0,0.0,0.0,True,False,missing_from_modeled
6,Burkholderia pseudomallei,163,227,47,0.0,0.0,0.0,True,False,missing_from_modeled
7,Chlamydia pneumoniae,45,47,21,0.0,0.0,0.0,True,False,missing_from_modeled
8,Chlamydia trachomatis,150,154,38,0.0,0.0,0.0,True,False,missing_from_modeled
9,Clostridium perfringens,265,412,67,0.0,0.0,0.0,True,False,missing_from_modeled


,raw_pathogen,raw_antibiotic,pathogen,antibiotic,pathogen_standardized,antibiotic_standardized,Class,Simplified categories,High level categories,Origin
0,Acinetobacter spp.,amikacin,Acinetobacter baumannii,Amikacin,True,True,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
1,Acinetobacter spp.,Colistin,Acinetobacter baumannii,Colistin,True,False,Polymyxins,Miscellaneous,Other,Naturally occurring/modified
2,Acinetobacter spp.,doripenem,Acinetobacter baumannii,Doripenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
3,Acinetobacter spp.,Gentamicin,Acinetobacter baumannii,Gentamicin,True,False,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
4,Acinetobacter spp.,Imipenem,Acinetobacter baumannii,Imipenem/cilastatin,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
5,Acinetobacter spp.,meropenem,Acinetobacter baumannii,Meropenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
6,Acinetobacter spp.,minocycline,Acinetobacter baumannii,Minocycline,True,True,Tetracyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
7,Acinetobacter spp.,Tigecycline,Acinetobacter baumannii,Tigecycline,True,False,Glycylcyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
8,E. coli,Cefepime,Escherichia coli,Cefepime,True,False,Fourth-generation-cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified
9,E. coli,cefotaxime,Escherichia coli,Cefotaxime,True,True,Third-generation-cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified


## Build Or Load Rank Data

In [23]:
rank_df = load_or_build_interactive_rank(
    si_df,
    RANK_OUTPUT_PATH,
    source_path=YEARLY_SI_PATH,
    fit_start_year=FIT_START_YEAR,
    fit_end_year=FIT_END_YEAR,
    curve_fit_end_year=CURVE_FIT_END_YEAR,
    num_future_years=NUM_FUTURE_YEARS,
    loess_frac=LOESS_FRAC,
    loess_degree=LOESS_DEGREE,
    tail_damping_years=TAIL_DAMPING_YEARS,
    verbose=True,
)
display(rank_df)

Loaded cached rank rows: 241 from /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/outputs/parquets/yearly_si_ranked_future.parquet


,pathogen,antibiotic,reference_year,future_year,num_future_years,fit_start_year,fit_end_year,curve_fit_end_year,post_fit_validation_end_year,raw_latest_available_year,...,tail_damping_years,validation_loess_frac,validation_loess_degree,validation_tail_damping_years,status,error,current_rank,future_rank,rank_change,abs_rank_change
0,Escherichia coli,Vancomycin,2025,2028,3,2000,2023,2025,2025,2026,...,None,0.6,1,None,ok,None,191.0,140.0,51.0,51.0
1,Neisseria gonorrhoeae,Azithromycin,2025,2028,3,2000,2023,2025,2025,2026,...,None,0.6,1,None,ok,None,40.0,84.0,-44.0,44.0
2,Pseudomonas aeruginosa,Vancomycin,2025,2028,3,2000,2023,2025,2025,2026,...,None,0.6,1,None,ok,None,185.0,144.0,41.0,41.0
3,Clostridioides difficile,Moxifloxacin,2025,2028,3,2000,2023,2025,2025,2026,...,None,0.6,1,None,ok,None,182.0,218.0,-36.0,36.0
4,Neisseria gonorrhoeae,Ceftriaxone,2025,2028,3,2000,2023,2025,2025,2026,...,None,0.6,1,None,ok,None,55.0,90.0,-35.0,35.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236,Haemophilus influenzae,Sulfamethoxazole,2025,2028,3,2000,2023,2025,2025,2026,...,None,0.6,1,None,ok,None,209.0,209.0,0.0,0.0
237,Haemophilus influenzae,Trimethoprim,2025,2028,3,2000,2023,2025,2025,2026,...,None,0.6,1,None,ok,None,211.0,211.0,0.0,0.0
238,Streptococcus pneumoniae,Chloramphenicol,2025,2028,3,2000,2023,2025,2025,2026,...,None,0.6,1,None,ok,None,235.0,235.0,0.0,0.0
239,Enterococcus faecium,Streptomycin,2025,2028,3,2000,2023,2025,2025,2026,...,None,0.6,1,None,ok,None,240.0,240.0,0.0,0.0


## Interactive Latest Count Vs Future SI Shift

In [24]:
latest_count_shift_result = plot_interactive_latest_observed_count_vs_future_si_shift(
    rank_df,
    glass_pairs_df=glass_pairs_df,
    min_last_available_count=MIN_LAST_AVAILABLE_COUNT,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    min_counts_background=MIN_COUNTS_BACKGROUND,
    min_abs_delta_si=MIN_ABS_DELTA_SI,
    min_si_delta=MIN_SI_DELTA,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    pdf_path=FIGURE_OUTPUT_DIR / "interactive_latest_observed_count_vs_future_si_shift.pdf",
    show=True,
)
latest_count_shift_df = latest_count_shift_result["plot_df"]
print(f"Displayed rows: {len(latest_count_shift_df):,}; GLASS matches: {int(latest_count_shift_df['is_glass_pair'].sum()):,}")
if latest_count_shift_result["pdf_path"]:
    print(f"Saved PDF: {latest_count_shift_result['pdf_path']}")
elif latest_count_shift_result.get("pdf_error"):
    print(f"PDF export failed: {latest_count_shift_result['pdf_error']}")

Displayed rows: 199; GLASS matches: 42
Saved PDF: /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/outputs/figures/interactive_latest_observed_count_vs_future_si_shift.pdf


## Display GLASS Matches In Plot

In [25]:
glass_plot_matches_df = build_glass_match_table(latest_count_shift_df)
display(glass_plot_matches_df)

,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean,glass_group_column
0,Escherichia coli,Ciprofloxacin,2025,2028,262.0,333.097360,0.617865,0.685116,0.067250,12.213819,antibiotic
1,Escherichia coli,Meropenem,2025,2028,136.0,264.693974,0.417391,0.572313,0.154922,20.320994,antibiotic
2,Klebsiella pneumoniae,Colistin,2025,2028,117.0,201.249824,0.395487,0.468664,0.073177,29.604378,antibiotic
3,Klebsiella pneumoniae,Meropenem,2025,2028,109.0,190.873451,0.310032,0.357864,0.047832,15.335670,antibiotic
4,Escherichia coli,Ceftriaxone,2025,2028,107.0,147.086165,0.433735,0.514628,0.080893,22.086599,antibiotic
5,Escherichia coli,Sulfamethoxazole/trimethoprim,2025,2028,105.0,159.395024,0.517313,0.546464,0.029151,16.819149,antibiotic
6,Pseudomonas aeruginosa,Meropenem,2025,2028,100.0,152.737863,0.318350,0.358318,0.039968,4.977528,antibiotic
7,Klebsiella pneumoniae,Ceftazidime,2025,2028,98.0,177.078096,0.238472,0.306324,0.067852,8.194002,antibiotic
8,Pseudomonas aeruginosa,Colistin,2025,2028,91.0,145.135975,0.273371,0.365695,0.092324,7.069436,antibiotic
9,Pseudomonas aeruginosa,Ceftazidime,2025,2028,91.0,138.228675,0.267938,0.296130,0.028192,13.478152,antibiotic


## Novel Non-GLASS Predictions

These are plotted pairs that do not appear in the cleaned GLASS list, sorted as candidate novel predictions of interest.

In [26]:
novel_prediction_df = build_novel_prediction_table(latest_count_shift_df)
print(f"Novel non-GLASS predictions displayed: {len(novel_prediction_df):,}")
display(novel_prediction_df)

Novel non-GLASS predictions displayed: 157


,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean
0,Escherichia coli,Vancomycin,2025,2028,29.0,55.697140,-0.094531,0.066697,0.161229,12.971636
1,Pseudomonas aeruginosa,Meticillin,2025,2028,76.0,114.054702,0.219351,0.354015,0.134664,17.823082
2,Escherichia coli,Doxycycline,2025,2028,32.0,50.299848,0.379792,0.512734,0.132942,7.863110
3,Escherichia coli,Tigecycline,2025,2028,57.0,82.597020,0.296958,0.429837,0.132878,9.367459
4,Escherichia coli,Azithromycin,2025,2028,27.0,38.036588,0.022266,0.154282,0.132016,45.119222
...,...,...,...,...,...,...,...,...,...,...
152,Enterococcus faecium,Tetracycline,2025,2028,18.0,23.449601,-0.091248,-0.151102,-0.059854,9.806837
153,Enterococcus faecium,Teicoplanin,2025,2028,11.0,8.056076,-0.065812,-0.132715,-0.066903,38.050148
154,Enterococcus faecalis,Gentamicin,2025,2028,13.0,13.860912,-0.200900,-0.273733,-0.072833,16.385757
155,Enterococcus faecalis,Tetracycline,2025,2028,22.0,25.841196,-0.045621,-0.126813,-0.081193,24.750722


## Future SI Vs Future SI Shift

This uses the same y-axis definition as the latest-count plot: predicted future SI minus smoothed SI at the latest available year.

In [27]:
future_si_shift_result = plot_future_si_vs_si_shift(
    latest_count_shift_df,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    pdf_path=FIGURE_OUTPUT_DIR / "future_si_vs_si_shift.pdf",
    show=True,
)
future_si_shift_fig = future_si_shift_result["figure"]
if future_si_shift_result["pdf_path"]:
    print(f"Saved PDF: {future_si_shift_result['pdf_path']}")
elif future_si_shift_result.get("pdf_error"):
    print(f"PDF export failed: {future_si_shift_result['pdf_error']}")

Saved PDF: /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/outputs/figures/future_si_vs_si_shift.pdf
